# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一段代码或技术问题
- **输出**：面向初学者的清晰解释（含示例、也可给更高效写法）
- **后端**：云端 `gpt-4o-mini` 与本地 `llama3.2`（经 Ollama 的 OpenAI 兼容接口）

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`，`base_url=http://localhost:11434/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地路径需 Ollama 已拉取 `llama3.2`
3. 在「提问」单元格改写 `question`，再跑 Ollama 回答格（GPT 流式格目前只有注释占位）


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display（流式刷新显示）
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：同一套 SDK 可打云端，也可打 Ollama 兼容端点
from openai import OpenAI


In [ ]:
# ========== 常量 + 两个客户端：云端 GPT 与本地 Ollama ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'
# Ollama 的 OpenAI 兼容 Base URL（注意是 /v1，不是原生 /api/chat）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 默认 OpenAI 客户端：密钥通常来自环境变量 OPENAI_API_KEY（需先 load_dotenv）
openai = OpenAI()
# 第二个客户端指向本地 Ollama；api_key 对本地服务多为占位，常用字符串 'ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== system prompt：定「模型以什么身份、怎么解释」==========

# system_prompt 保留英文：这是发给模型的指令，改译会改变回答风格/行为
system_prompt = """
You are a professional software coding master. 
You will help explain an input of code.
Explain how this code works with an example.
Also suggest diffrent ways of writing this code efficiently if there is an alternative.
Respond to a user who is a beginner. """


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再跑下面的回答单元格做对比
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 路径 A（占位）：本来要用 gpt-4o-mini 做流式回答 ==========
# 本格目前只有说明注释、没有可执行调用；作者意图是「带 streaming」接听 GPT。
# 若你要补全：可参考下方 ollama 版，把 client 换成 openai、model 换成 MODEL_GPT，并设 stream=True。
# 让 gpt-4o-mini 接听，带流媒体


In [ ]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）一次性回答并 Markdown 展示 ==========

# 定义函数：接收用户问题，调用本地兼容接口，把完整回答渲染成 Markdown
def code_examiner_ollama(question):
    # chat.completions.create：非流式；等整段生成完再取 message.content
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            # system：角色与答题风格；user：具体问题
            {"role": "system", "content":system_prompt},
            {"role": "user", "content": question}
        ],
    )
    # 从 choices[0] 取出助手回复文本
    result = response.choices[0].message.content
    # 在笔记本里用 Markdown 漂亮显示完整回答
    display(Markdown(result))


In [ ]:
# ========== 调用：把上面的 question 交给 Ollama 解释器 ==========

# 运行本格即可看到本地 llama3.2 对 question 的解释
code_examiner_ollama(question)
